In [55]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
import numpy as np
from itertools import product
from statsmodels.tsa.arima.model import ARIMA

In [34]:
def load_data():
    df = pd.read_csv("../data/data_preprocessed/processed_data.csv")
    df = df.set_index(["Country", "Year"])
    return df

In [35]:
df = load_data()

In [36]:
df

Capacity  Economic  Ecosystems  Exposure      Food  Governance  \
Country Year                                                                   
AFG     1995  0.893281  0.496497    0.515884  0.480512  0.665745    0.128090   
        1996  0.893892  0.496497    0.516995  0.480512  0.665583    0.128090   
        1997  0.894459  0.496497    0.518030  0.480512  0.665420    0.141392   
        1998  0.890161  0.496497    0.518156  0.480512  0.657103    0.154694   
        1999  0.885896  0.496497    0.518156  0.480512  0.648959    0.143689   
...                ...       ...         ...       ...       ...         ...   
ZWE     2019  0.597433  0.250663    0.515729  0.516912  0.544129    0.128405   
        2020  0.591558  0.260265    0.515480  0.516912  0.571499    0.124162   
        2021  0.596365  0.260265    0.515921  0.516912  0.560939    0.131438   
        2022  0.583779  0.260265    0.513962  0.516912  0.557942    0.135388   
        2023  0.573759  0.260265    0.513962  0.516912  0.559073    0.135101   

               Habitat    Health  Infrastructure  Readiness  Sensitivity  \
Country Year                                                               
AFG     1995  0.602933  0.748302        0.355586   0.308182     0.419587   
        1996  0.604235  0.748302        0.354484   0.308181     0.419307   
        1997  0.605583  0.748302        0.353925   0.312614     0.419055   
        1998  0.606909  0.748302        0.353710   0.317047     0.418794   
        1999  0.608137  0.748302        0.352292   0.313378     0.419433   
...                ...       ...             ...        ...          ...   
ZWE     2019  0.571407  0.699729        0.320293   0.170180     0.419507   
        2020  0.564062  0.699319        0.305472   0.172504     0.432905   
        2021  0.556834  0.700294        0.306890   0.177080     0.420585   
        2022  0.549853  0.699798        0.299974   0.178652     0.424039   
        2023  0.548898  0.699798        0.275926   0.178869     0.424127   

                Social  Vulnerability     Water  
Country Year                                     
AFG     1995  0.299958       0.612511  0.529692  
        1996  0.299954       0.612679  0.528281  
        1997  0.299952       0.612837  0.526853  
        1998  0.299949       0.611179  0.525424  
        1999  0.299947       0.609828  0.525585  
...                ...            ...       ...  
ZWE     2019  0.131472       0.505822  0.383647  
        2020  0.133086       0.507918  0.391679  
        2021  0.139537       0.505453  0.391842  
        2022  0.140304       0.502217  0.391776  
        2023  0.141240       0.498239  0.391776  

[5394 rows x 14 columns]

In [37]:
indicators = df.columns
countries = df.index.get_level_values("Country").unique()

In [44]:
results = []

for country in countries:
    for indicator in indicators:

        series = df.loc[country, indicator].dropna()

        try:
            p_value = adfuller(series)[1]
        except ValueError:
            p_value = None

        results.append({
            "Country": country,
            "Indicator": indicator,
            "p_value": p_value
        })

adf_results = pd.DataFrame(results)

/Users/deniz/.pyenv/versions/climate-resilience-dashboard/lib/python3.10/site-packages/statsmodels/regression/linear_model.py:955: RuntimeWarning: divide by zero encountered in log
  llf = -nobs2*np.log(2*np.pi) - nobs2*np.log(ssr / nobs) - nobs2
/Users/deniz/.pyenv/versions/climate-resilience-dashboard/lib/python3.10/site-packages/statsmodels/regression/linear_model.py:955: RuntimeWarning: divide by zero encountered in log
  llf = -nobs2*np.log(2*np.pi) - nobs2*np.log(ssr / nobs) - nobs2
/Users/deniz/.pyenv/versions/climate-resilience-dashboard/lib/python3.10/site-packages/statsmodels/regression/linear_model.py:955: RuntimeWarning: divide by zero encountered in log
  llf = -nobs2*np.log(2*np.pi) - nobs2*np.log(ssr / nobs) - nobs2
/Users/deniz/.pyenv/versions/climate-resilience-dashboard/lib/python3.10/site-packages/statsmodels/regression/linear_model.py:955: RuntimeWarning: divide by zero encountered in log
  llf = -nobs2*np.log(2*np.pi) - nobs2*np.log(ssr / nobs) - nobs2


In [45]:
adf_table = adf_results.pivot(
    index="Country",
    columns="Indicator",
    values="p_value"
)

In [53]:
(adf_table > 0.05).sum()

Indicator
Capacity          165
Economic          165
Ecosystems        115
Exposure            0
Food              157
Governance        156
Habitat           151
Health            168
Infrastructure    170
Readiness         166
Sensitivity       151
Social            162
Vulnerability     178
Water             146
dtype: int64

### p > 0.05 → fail to reject the null hypothesis → the series is likely non-stationary.

In [56]:
best_aic = float("inf")
best_order = None
best_model = None

In [108]:
def select_order(series, p_range, d_range, q_range):

    best_aic = float("inf")
    best_order = None

    for p, d, q in product(p_range, d_range, q_range):
        try:
            model = ARIMA(series, order=(p, d, q)).fit()

            if model.aic < best_aic:
                best_aic = model.aic
                best_order = (p, d, q)

        except Exception:
            continue

    return best_order

In [109]:
best_order

(1, 0, 2)

In [110]:
best_aic

np.float64(-272.15731475921035)

In [111]:
print(country, indicator)
print(order)
print(type(order))

AFG Capacity
((3, 0, 1), np.float64(-167.25427486141848))
<class 'tuple'>


In [ ]:
results = []

for country in countries:
    for indicator in indicators:

        series = df.loc[country, indicator]

        order = select_order(
            series,
            p_range=range(4),
            d_range=range(3),
            q_range=range(4),
        )

        if order is None:
            print(f"Skipping {country} - {indicator}")
            continue

        model = ARIMA(series, order=order).fit()

        results.append({
            "Country": country,
            "Indicator": indicator,
            "Order": order,
            "AIC": model.aic,
            "BIC": model.bic,
        })

results_df = pd.DataFrame(results)
print(results_df.head())

/Users/deniz/.pyenv/versions/climate-resilience-dashboard/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/Users/deniz/.pyenv/versions/climate-resilience-dashboard/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/Users/deniz/.pyenv/versions/climate-resilience-dashboard/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/Users/deniz/.py

In [ ]:
print(model.model.order)

In [ ]:
results_df = pd.DataFrame(results)

In [ ]:
results_df